# Voice Screening Agent - Colab runner

A voice agent that runs a short senior .NET technical screening end to end:
speak an answer, it transcribes (Egyptian Arabic + English code-mixing handled),
retrieves the relevant rubric criteria, **decides whether to ask one clarifying
follow-up**, then scores 1-5 and speaks the result back in your language.

Everything is open-source and self-hosted. No API keys for the pipeline itself.

**Before you start:** set the runtime to a GPU.
`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU`

Then `Runtime -> Run all`. First run takes ~12-15 minutes, almost all of it
downloading ~10 GB of model weights.

### Two things about the ordering, because they are not arbitrary

**The LLM is loaded onto the GPU first**, before Whisper and BGE-M3. All four
models together need ~11 GB of the T4's 15 GB, which fits - but only if nothing
loads twice. If Whisper and BGE-M3 claim VRAM first and something else takes a
second copy, Ollama gets squeezed off the GPU and silently falls back to CPU,
where a 7B model generates at single-digit tokens/sec and every request times
out.

**Everything runs in this kernel, not as `!python script.py`.** A subprocess
loads its own copy of Whisper and BGE-M3 - that is exactly the second copy that
breaks things.

The UI launch is the **last** cell, because it blocks forever and nothing below
it would ever run.


## 1. Confirm the GPU

In [1]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU"
)

def vram(tag=""):
    free, total = torch.cuda.mem_get_info()
    print(f"[VRAM] {tag:<28} {(total - free) / 1e9:5.1f} / {total / 1e9:.1f} GB used")

print(f"\n{torch.cuda.get_device_name(0)}")
vram("baseline")

Mon Jul 27 12:49:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repository

In [2]:
import os, pathlib

REPO_URL = "https://github.com/karimgamalmahmoud/Voice-Agentic-Systems.git"
REPO_DIR = pathlib.Path("/content/Voice-Agentic-Systems")

if REPO_DIR.exists():
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("\nWorking directory:", os.getcwd())

Cloning into '/content/Voice-Agentic-Systems'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 86 (delta 24), reused 81 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 2.11 MiB | 4.96 MiB/s, done.
Resolving deltas: 100% (24/24), done.

Working directory: /content/Voice-Agentic-Systems


## 3. Install Python dependencies

Torch is deliberately left alone - Colab's build is already CUDA-matched, and
reinstalling it is the fastest way to break the runtime.

If pip prints a "You must restart the runtime" banner, do it, then
`Runtime -> Run all` again. Cell 2 is a no-op on the second pass.

In [3]:
!pip install -q -r requirements.txt

import transformers, gradio, sentence_transformers
print("transformers", transformers.__version__)
print("gradio", gradio.__version__)
print("sentence-transformers", sentence_transformers.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.7/930.7 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.1 MB/s eta 0:00:00
transformers 4.57.6
gradio 6.17.3
sentence-transformers 5.6.0


## 4. Install and start Ollama

Ollama serves the LLM behind an OpenAI-compatible API in a separate process,
which keeps its dependencies away from the torch / Whisper / TTS stack.

In [4]:
# Ollama's installer unpacks a zstd-compressed archive and the Colab image does
# not ship zstd. Without this the install dies with
# "ERROR: This version requires zstd for extraction".
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)

!curl -fsSL https://ollama.com/install.sh | sh

# Fail here rather than three cells later with a confusing connection error
# against 127.0.0.1:11434, which points at entirely the wrong problem.
import shutil
assert shutil.which("ollama"), "Ollama did not install - check the output above"
!ollama --version

Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
import subprocess, time, os, requests

# KEEP_ALIVE has to be set on the server, not per request. Without it Ollama
# unloads the model after five idle minutes; reloading it later, once Whisper
# and BGE-M3 hold VRAM, is exactly when it fails to fit and drops to CPU.
env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_KEEP_ALIVE": "60m"}

log = open("/content/ollama.log", "w")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=log,
                          stderr=subprocess.STDOUT)

for attempt in range(60):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok:
            print(f"Ollama up after {attempt + 1}s")
            break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/ollama.log").read()[-2000:])
    raise RuntimeError("Ollama did not start - log above")

Ollama up after 3s


### Pull the model

`qwen2.5:7b-instruct` (~4.7 GB). The strongest Arabic model that fits a free T4
alongside Whisper and BGE-M3, and dependable at the structured JSON the coverage
and scoring stages need.

In [6]:
!ollama pull qwen2.5:7b-instruct

### Load it onto the GPU now, and verify it actually landed there

This is the cell that prevents the failure mode described at the top. A real
generation request forces Ollama to load the weights, so it claims its ~5.5 GB
**before** Whisper and BGE-M3 take theirs.

`ollama ps` must show **100% GPU** in the PROCESSOR column. If it says CPU, stop
and restart the runtime - everything downstream will time out.

In [7]:
import sys, time
sys.path.insert(0, "/content/Voice-Agentic-Systems/src")

from voice_agent.llm import LLMClient

llm = LLMClient()
t0 = time.time()
reply = llm.complete("Answer with exactly one word.", "Say READY.", max_tokens=8)
print(f"LLM replied {reply!r} in {time.time() - t0:.1f}s")

!ollama ps
vram("after LLM load")

LLM replied 'READY.' in 110.4s
NAME                   ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
qwen2.5:7b-instruct    845dbda0ea48    4.7 GB    100% GPU     4096       59 minutes from now    
[VRAM] after LLM load                 5.0 / 15.6 GB used


In [8]:
# Rough throughput check. Under ~5 tok/s the model is on CPU, not the GPU,
# and the pipeline below will crawl.
t0 = time.time()
out = llm.complete("You are terse.", "Count from 1 to 40, space separated.", max_tokens=200)
rate = len(out.split()) / (time.time() - t0)
print(f"~{rate:.1f} tokens/sec")
print("OK - on GPU" if rate > 5 else "TOO SLOW - model is on CPU, restart the runtime")

~9.4 tokens/sec
OK - on GPU


## 5. Unit tests

CPU-only, a couple of seconds. Covers the follow-up decision policy, corpus
chunking and code-mixed language detection. If these fail, the problem is the
checkout, not the GPU.

In [9]:
!python -m pytest tests/ -q

.............................................                            [100%]
45 passed in 0.31s


## 6. Load Whisper and BGE-M3

Whisper large-v3 (~3 GB) and BGE-M3 (~2.2 GB), into **this** kernel. Everything
below reuses this one agent instance. Expect ~4-6 minutes on a cold cache.

In [10]:
import logging
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

from voice_agent.agent import ScreeningAgent

agent = ScreeningAgent()
agent.warm_up()

ok, msg = agent.llm.health()
print("LLM:", msg)
assert ok, msg
vram("all models resident")

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

LLM: qwen2.5:7b-instruct ready at http://127.0.0.1:11434/v1
[VRAM] all models resident           10.9 / 15.6 GB used


## 7. Transcription check

The riskiest part of this stack is Egyptian Arabic mixed with English technical
terms, so eyeball the transcripts before anything else. Two of the three
provided samples are code-mixed.

Look for Arabic script for the Arabic, and English technical terms
("async", "EF Core", "thread pool") surviving in Latin script rather than being
transliterated into Arabic.

In [11]:
import time
from voice_agent.config import AUDIO_DIR

transcripts = {}
for path in sorted(AUDIO_DIR.glob("*.mp3")):
    t0 = time.time()
    tr = agent.transcribe(str(path))
    transcripts[path.name] = tr
    print(f"\n{'=' * 70}\n{path.name}")
    print(f"  lang={tr.language}  arabic={tr.arabic_ratio:.0%}  "
          f"audio={tr.duration_s:.0f}s  took={time.time() - t0:.0f}s")
    print(f"{'=' * 70}\n{tr.text}")


answer_1.mp3
  lang=en  arabic=0%  audio=55s  took=17s
Alright, so the first thing I would do is not change anything yet. I would reproduce the problem and measure it. I would look at our EPM and our logs to see where the time actually goes. Is it database? Is it an external call or CPU? Most of the time under the load, it is a data layer, so I would check the SQL we are generating. I would look for an N plus 1 pattern or a missing index. And since this endpoint sounds read-only, I would add as no tracking to skip the tracking overhead. Also, if the data is read-heavy and we can't tolerate being slightly stale, I would add in-memory cache first and then Redis if we are running multiple instances with clear rule for invalidating GAT then finally I will load test to confirm the P95

answer_2.mp3
  lang=ar  arabic=86%  audio=37s  took=14s
طيب انا هبدأ بالداتابيز غالبا لان دي اكتر حاجة بتبقى بطيئة هبص على ال end point بيعملها يمكن في كويري بيتنفذ اكتر من مرة او ممكن ما يكونش عليها اندكت ك

## 8. Quality gate - the full pipeline over all three samples

The end-to-end proof, and a required deliverable. Each sample runs
transcribe -> retrieve -> assess coverage -> branch -> score, and the invariants
are checked: the two irrelevant reference notes stay out of retrieval, Arabic in
produces Arabic out, and scores have not drifted from the stored baseline.

Run **in this kernel against the already-loaded agent** - see the note at the
top about why a subprocess breaks this.

Writes `docs/QUALITY_GATE_RESULTS.md`. The first run creates the baseline, so it
cannot fail on drift; later runs compare against it.

This is the slow cell: three samples, two or three LLM calls each. Allow
5-10 minutes.

In [12]:
sys.path.insert(0, "/content/Voice-Agentic-Systems/scripts")
from run_quality_gate import run_gate

exit_code = run_gate(agent=agent, speak=False)   # TTS exercised separately below
print("\nexit code:", exit_code)

LLM: qwen2.5:7b-instruct ready at http://127.0.0.1:11434/v1

=== answer_1.mp3 ===


  lang=en follow_up=True score=4/5
  Candidate systematically addresses the issue by measuring before changing, covers DB optimization (N+1 pattern, missing index), async/await (as no tracking), and caching. However, misses discussing sync-over-async issues and cancellation tokens.

=== answer_2.mp3 ===
  lang=ar follow_up=True score=3/5
  يقول إن يبدأ بالداتابيز ويصيغ حلول للاستيريشن و الكاشينج لكنه لا يذكر الطريقة النظامية للمعاينة والتعديل.

=== answer_3.mp3 ===
  lang=ar follow_up=False score=2/5
  أجاب على بعض الجوانب مثل إعادة تشغيل الخادم والموارد، لكنه لم يوضح كيفية التحقق من المشكلة أو استخدام أدوات التشخيص. لم يذكر الـ async/await أو الـ EF Core بشكل صحيح، ولم يتحدث عن الاستراتيجيات المحددة للاستناد إلى الدخول أو التخزين المؤقت.

Baseline written to tests/golden/quality_gate.json

Report: docs/QUALITY_GATE_RESULTS.md

✅ QUALITY GATE PASSED

exit code: 0


In [13]:
from IPython.display import Markdown, display
display(Markdown(open("docs/QUALITY_GATE_RESULTS.md", encoding="utf-8").read()))

# Quality Gate Results

Generated by `scripts/run_quality_gate.py`. Each provided sample answer is run
through the full pipeline: transcribe → retrieve → assess coverage → branch → score.

In batch mode no human is present to answer a clarifying question, so when the
agent decides to follow up, the question it composed is recorded and scoring
falls back to the original answer. The branch is still exercised and logged.

| Sample | Lang | Followed up | Score | Justification |
|---|---|---|---|---|
| answer_1.mp3 | `en` | yes | **4/5** | Candidate systematically addresses the issue by measuring before changing, covers DB optimization (N+1 pattern, missing index), async/await (as no tracking), and caching. However, misses discussing sync-over-async issues and cancellation tokens. |
| answer_2.mp3 | `ar` | yes | **3/5** | يقول إن يبدأ بالداتابيز ويصيغ حلول للاستيريشن و الكاشينج لكنه لا يذكر الطريقة النظامية للمعاينة والتعديل. |
| answer_3.mp3 | `ar` | no | **2/5** | أجاب على بعض الجوانب مثل إعادة تشغيل الخادم والموارد، لكنه لم يوضح كيفية التحقق من المشكلة أو استخدام أدوات التشخيص. لم يذكر الـ async/await أو الـ EF Core بشكل صحيح، ولم يتحدث عن الاستراتيجيات المحددة للاستناد إلى الدخول أو التخزين المؤقت. |

---

## answer_1.mp3

- **Language detected:** `en` (Arabic script 0%) · 54.8s
- **Competencies retrieved:** Data access (EF Core) `0.6512`, Async & concurrency `0.5925`, Diagnostic method `0.566`, Caching & performance `0.5422`
- **Reference notes retrieved:** 01_async_concurrency `0.6648`, 03_data_access_ef_core `0.6619`

**Transcript**

> Alright, so the first thing I would do is not change anything yet. I would reproduce the problem and measure it. I would look at our EPM and our logs to see where the time actually goes. Is it database? Is it an external call or CPU? Most of the time under the load, it is a data layer, so I would check the SQL we are generating. I would look for an N plus 1 pattern or a missing index. And since this endpoint sounds read-only, I would add as no tracking to skip the tracking overhead. Also, if the data is read-heavy and we can't tolerate being slightly stale, I would add in-memory cache first and then Redis if we are running multiple instances with clear rule for invalidating GAT then finally I will load test to confirm the P95

**Coverage**

- Data access (EF Core): `covered`
- Async & concurrency: `partial`
- Diagnostic method: `covered`
- Caching & performance: `covered`

**Branch decision:** ask a follow-up

> 'Async & concurrency' is partially covered (confidence 0.70, priority 0.95) and is the highest-value resolvable gap of 1 candidate(s). Gap: The candidate did not discuss async/await patterns, sync-over-async issues, or the use of cancellation tokens..

**Follow-up composed:** كيف ستتعامل مع سيناريو التعادل بين استخدام await وال Mutex للتحكم في متزامنية العمليات أثناء التحقيق؟

**Score:** 4/5 — Candidate systematically addresses the issue by measuring before changing, covers DB optimization (N+1 pattern, missing index), async/await (as no tracking), and caching. However, misses discussing sync-over-async issues and cancellation tokens.

- Covered: measures before changing, DB optimization, as no tracking, caching
- Missed: sync-over-async issues, cancellation tokens

## answer_2.mp3

- **Language detected:** `ar` (Arabic script 86%) · 37.5s
- **Competencies retrieved:** Data access (EF Core) `0.6336`, Async & concurrency `0.624`, Caching & performance `0.5987`, Diagnostic method `0.5431`
- **Reference notes retrieved:** 01_async_concurrency `0.6966`, 03_data_access_ef_core `0.6525`

**Transcript**

> طيب انا هبدأ بالداتابيز غالبا لان دي اكتر حاجة بتبقى بطيئة هبص على ال end point بيعملها يمكن في كويري بيتنفذ اكتر من مرة او ممكن ما يكونش عليها اندكت كويس ممكن برضو اضيف caching بالredis لو ال data ready read heavy عشان نخفف الضغط بالنسبسبة يعني المفروض نخلي عشان ما يحصلش بس صراحة مش متأكد قوي بيأثر زي تحت اللوت. عموما تبقى اول حاجة ارجح.

**Coverage**

- Data access (EF Core): `covered`
- Async & concurrency: `missing`
- Caching & performance: `covered`
- Diagnostic method: `partial`

**Branch decision:** ask a follow-up

> 'Diagnostic method' is partially covered (confidence 0.80, priority 1.00) and is the highest-value resolvable gap of 1 candidate(s). Gap: measuring before changing and gathering evidence (APM, logs, profiler).

**Follow-up composed:** كيف هتقدر تعرف إن ال problem موجود في الداتابيز أم في الأندكت؟

**Score:** 3/5 — يقول إن يبدأ بالداتابيز ويصيغ حلول للاستيريشن و الكاشينج لكنه لا يذكر الطريقة النظامية للمعاينة والتعديل.

- Covered: الداتابيز, استيريشن, كاشينج
- Missed: الطريقة النظامية للمعاينة والتعديل

## answer_3.mp3

- **Language detected:** `ar` (Arabic script 87%) · 37.9s
- **Competencies retrieved:** Async & concurrency `0.6234`, Data access (EF Core) `0.6231`, Diagnostic method `0.5416`, Caching & performance `0.5213`
- **Reference notes retrieved:** 01_async_concurrency `0.6957`, 03_data_access_ef_core `0.6277`

**Transcript**

> طيب يعني هو لو انا عندي end point بطيئة اول حاجة اعملها ممكن ارستارت السيرفر واشوف المشكلة هتتحل ولا لا لو لسه بطيئ فممكن يبقى امكانيات السيرفر قليلة فمحتاجة تزود رمايات مثلا او نجيب سيرفر اقوى برضو ممكن يكون في حتة في كود مكتوبة بشكل مش كويس محتاجة تتزبط صراحة مش عارف الـ Entity Framework أو الـ Database Queries ممكن يؤثروا زي في الموضوع ده بس في الغالب لو زودنا الإمكانيات بتاع الـ Server فمفروض الدنيا هتتحسن

**Coverage**

- Async & concurrency: `missing`
- Data access (EF Core): `missing`
- Diagnostic method: `partial`
- Caching & performance: `missing`

**Branch decision:** move on and score

> 3/4 competencies are absent (75% >= 60% threshold). The answer is thin across the board; one clarification cannot change the band.

**Score:** 2/5 — أجاب على بعض الجوانب مثل إعادة تشغيل الخادم والموارد، لكنه لم يوضح كيفية التحقق من المشكلة أو استخدام أدوات التشخيص. لم يذكر الـ async/await أو الـ EF Core بشكل صحيح، ولم يتحدث عن الاستراتيجيات المحددة للاستناد إلى الدخول أو التخزين المؤقت.

- Covered: إعادة تشغيل الخادم
- Missed: قياس المشكلة, استخدام أدوات التشخيص, الـ async/await, الـ EF Core

---

## Gate status

✅ All checks passed.

## How I'd know a change made this worse

The gate pins the things that should never move and tolerates the one thing that
legitimately does. Retrieval precision is binary: the two irrelevant reference notes
(dependency injection, API security) must stay out of the retrieved set, and if a
chunking or embedding change lets them in, retrieval has stopped discriminating.
Language fidelity is likewise binary: an Arabic answer that comes back scored in
English is a regression no score comparison would catch. Scores themselves I treat as
noisy, so they are compared against a stored baseline with a ±1 tolerance — a single
sample drifting one point is sampling noise, but two samples moving together, or any
sample moving two points, means the prompt or model change altered the scoring band
and needs a human to look. The branch decisions are also snapshotted: if a sample that
used to earn a follow-up stops earning one, the coverage pass has changed its
partial/missing calibration even when the final score happens to land the same.

## 9. Check the Arabic speak-back

TTS is the weakest component in an all-open-source stack, so confirm it makes
sound before relying on it live. If this is silent the loop still works - the UI
falls back to text - but you want to know now.

In [14]:
from IPython.display import Audio, display

sample = agent.tts.synthesize("تقييمك 4 من 5. إجابة كويسة بس ناقصها تفاصيل.", "ar")
if sample is None:
    print("Arabic TTS unavailable - the UI will show text only.")
else:
    rate, wav = sample
    print(f"Arabic OK: {len(wav) / rate:.1f}s")
    display(Audio(wav, rate=rate))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

Arabic OK: 7.1s


### Fallback: just download the report

If you would rather not put a key in the notebook at all, grab the file and
commit it from your machine.

In [17]:
from google.colab import files
files.download("docs/QUALITY_GATE_RESULTS.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Launch the app  ← last cell, blocks while running

Prints a public `gradio.live` URL. Open it in a **new tab** - the microphone
works through that tunnel, which is what makes this runnable with no local
install. Grant mic permission on the `gradio.live` origin, not on the Colab tab.

The link stays alive while this cell runs. Stop the cell to shut it down.

In [ ]:
from voice_agent.app import build_ui
import voice_agent.app as app_module

app_module.AGENT = agent      # reuse the models already loaded in cell 6
build_ui().launch(share=True)

---

### Troubleshooting

**Every LLM call times out** - the model is on CPU, not GPU. Check `ollama ps`
in cell 4: the PROCESSOR column must read 100% GPU. The usual cause is a second
copy of Whisper or BGE-M3 in VRAM, which happens if you run the scripts as
`!python scripts/...` instead of the in-kernel cells above. `Runtime -> Restart
session` and run in order.

**`ERROR: This version requires zstd for extraction`** - the Ollama installer
unpacks a zstd archive and the Colab image does not ship zstd. Cell 4 installs
it first.

**`FileNotFoundError: docs/QUALITY_GATE_RESULTS.md`** - the gate cell above it
failed, so nothing was written. Scroll up and read that error.

**Gradio link not appearing** - the last cell must stay running. If it errored,
rerun the Ollama start cell; Colab sometimes reaps the process.

**CUDA out of memory** - `Runtime -> Restart session`, then run all in order.
The four models total ~11 GB of the T4's 15 GB, which fits only if nothing
loads twice.

**Arabic speak-back is silent** - MMS-TTS needs romanized input via the `uroman`
package. If that failed to install, TTS degrades to text-only by design and the
rest of the loop is unaffected.

**Microphone blocked** - the browser needs permission on the `gradio.live`
origin, not on the Colab tab. Look for the mic icon in the address bar.
